In [21]:
import unicodedata
import re
import html

import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import load_npz
import joblib
import numpy as np
import nltk
from IPython.display import display
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

DATA_DIR_PATH = "../data/"

In [22]:
INPUT_PATH = DATA_DIR_PATH + "preprocessed/normalized_dataset.csv"
INPUT_VECTOR_PATH = DATA_DIR_PATH + "artifacts/tfidf_vectorizer.joblib"
INPUT_MATRIX_PATH = DATA_DIR_PATH + "artifacts/tfidf_matrix.npz"

In [23]:
df = pd.read_csv(INPUT_PATH)

df.head()

,link,date,title,content,keywords,category,normalized_title,normalized_content,normalized_keywords,normalized,tokens,filtered_tokens
0,https://entertainment.kompas.com/read/2026/07/30/092657666/deretan-aktor-yang-pernah-perankan-spider-man,2026-07-30 09:26:57,Deretan Aktor yang Pernah Perankan Spider-Man,"JAKARTA, KOMPAS.com- KehadiranSpider-Man: Brand New Dayyang tayang mulai 29 Juli 2026 menjadi momentum tepat untuk menengok kembali perjalanan panjang sosok Peter Parker di layar lebar. Film garapan Destin Daniel Cretton ini menjadi babak keempat petualanganSpider-ManversiTom Hollanddalam Marvel Cinematic Universe (MCU), sekaligus menandai era baru setelah trilogi sebelumnya rampung digarap sutradara Jon Watts. Baca juga:REVIEW: Spider-Man: Brand New Day, Kisah Pendewasaan Sang Manusia Laba-...","peter parker, mcu, spider-man, tom holland",Hype,deretan aktor yang pernah perankan spider man,jakarta kehadiranspider man brand new dayyang tayang mulai juli menjadi momentum tepat untuk menengok kembali perjalanan panjang sosok peter parker di layar lebar film garapan destin daniel cretton ini menjadi babak keempat petualanganspider manversitom hollanddalam marvel cinematic universe mcu sekaligus menandai era baru setelah trilogi sebelumnya rampung digarap sutradara jon watts baca juga review spider man brand new day kisah pendewasaan sang manusia laba laba namun jauh sebelum tom ho...,peter parker mcu spider man tom holland,deretan aktor yang pernah perankan spider man jakarta kehadiranspider man brand new dayyang tayang mulai juli menjadi momentum tepat untuk menengok kembali perjalanan panjang sosok peter parker di layar lebar film garapan destin daniel cretton ini menjadi babak keempat petualanganspider manversitom hollanddalam marvel cinematic universe mcu sekaligus menandai era baru setelah trilogi sebelumnya rampung digarap sutradara jon watts baca juga review spider man brand new day kisah pendewasaan sa...,"['deretan', 'aktor', 'yang', 'pernah', 'perankan', 'spider', 'man', 'jakarta', 'kehadiranspider', 'man', 'brand', 'new', 'dayyang', 'tayang', 'mulai', 'juli', 'menjadi', 'momentum', 'tepat', 'untuk', 'menengok', 'kembali', 'perjalanan', 'panjang', 'sosok', 'peter', 'parker', 'di', 'layar', 'lebar', 'film', 'garapan', 'destin', 'daniel', 'cretton', 'ini', 'menjadi', 'babak', 'keempat', 'petualanganspider', 'manversitom', 'hollanddalam', 'marvel', 'cinematic', 'universe', 'mcu', 'sekaligus', '...","['deretan', 'aktor', 'perankan', 'spider', 'man', 'jakarta', 'kehadiranspider', 'man', 'brand', 'new', 'dayyang', 'tayang', 'juli', 'momentum', 'menengok', 'perjalanan', 'sosok', 'peter', 'parker', 'layar', 'lebar', 'film', 'garapan', 'destin', 'daniel', 'cretton', 'babak', 'keempat', 'petualanganspider', 'manversitom', 'hollanddalam', 'marvel', 'cinematic', 'universe', 'mcu', 'menandai', 'era', 'trilogi', 'rampung', 'digarap', 'sutradara', 'jon', 'watts', 'baca', 'review', 'spider', 'man', ..."
1,https://www.kompas.com/hype/read/2026/08/04/165109866/aktor-ryotaro-shimizu-ditemukan-meninggal-dunia-di-kediamannya,2026-08-04 16:51:09,Aktor Ryotaro Shimizu Ditemukan Meninggal Dunia di Kediamannya,"KOMPAS.com- Aktor Ryotaro Shimizu, yang karier hiburannya hancur karena vonis kasus narkoba, meninggal dunia pada usia 37 tahun. Penyebab kematian belum diumumkan tetapi menurut media Jepang, termasuk TV Asahi,Ryotaro Shimizuditemukan tidak sadarkan diri di rumahnya pada tanggal 1 Agustus. Petugas medis darurat tiba setelah dihubungi oleh manajernya, tetapi ia dinyatakan meninggal di tempat kejadian. Polisi menyatakan bahwa tidak ditemukan tanda-tanda tindak kejahatan dan saat ini sedang men...","ryotaro shimizu, ryotaro shimizu meninggal",Hype,aktor ryotaro shimizu ditemukan meninggal dunia di kediamannya,aktor ryotaro shimizu yang karier hiburannya hancur karena vonis kasus narkoba meninggal dunia pada usia tahun penyebab kematian belum diumumkan tetapi menurut media jepang termasuk tv asahi ryotaro shimizuditemukan tidak sadarkan diri

In [24]:
matrix = load_npz(INPUT_MATRIX_PATH)
vectorizer = joblib.load(INPUT_VECTOR_PATH)

In [25]:
# Normalize Text
def normalize_text(text, keep_numbers = False):
    if pd.isna(text):
        return ""

    text = str(text)

    # Normalize Unicode representation first
    text = unicodedata.normalize("NFKC", text)

    # Decode HTML entities
    text = html.unescape(text)

    # Remove HTML tags
    text = re.sub(r"<[^>]+>", " ", text)

    # Remove URLs
    text = re.sub(
        r"(?:https?://|www\.)\S+",
        " ",
        text,
        flags=re.IGNORECASE
    )

    # Remove standalone domains
    text = re.sub(
        r"\b(?:[\w-]+\.)+(?:com|co\.id|id|net|org|go\.id|ac\.id)\b",
        " ",
        text,
        flags=re.IGNORECASE
    )

    # Remove email addresses
    text = re.sub(
        r"\b[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}\b",
        " ",
        text
    )

    # Remove @mentions
    text = re.sub(
        r"@[A-Za-z0-9_]+",
        " ",
        text
    )

    # Lowercase
    text = text.lower()

    # Keep Unicode letters, digits, and whitespace
    # Remove punctuation and special characters
    text = re.sub(r"[^\w\s]", " ", text)

    # Remove underscores
    text = re.sub(r"_", " ", text)

    # Optionally remove numbers
    if not keep_numbers:
        text = re.sub(r"\d+", " ", text)

    # Normalize elongated characters
    # Example: goooood -> good
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [26]:
# Stopword Removal
stopwords_id = set(stopwords.words("indonesian"))
stopwords_en = set(stopwords.words("english"))

stop_words = stopwords_id.union(stopwords_en)

In [27]:
def preprocess_query(query):
  normalized_query = normalize_text(query)
  tokens = word_tokenize(normalized_query)
  filtered_tokens = [token for token in tokens if token not in stop_words and len(token) > 1]
  processed_query = " ".join(filtered_tokens)

  return {
    "normalized_query": normalized_query,
    "tokens": tokens,
    "filtered_tokens": filtered_tokens,
    "processed_query": processed_query
  }

In [28]:
def extract_expansion_terms(
    tfidf_matrix,
    tfidf_vectorizer,
    top_indices,
    similarity_scores,
    query_tokens,
    top_n=5
):
    # Normalize feedback scores
    if similarity_scores.sum() == 0:
        feedback_weights = (
            np.ones(len(top_indices)) / len(top_indices)
        )
    else:
        feedback_weights = (
            similarity_scores / similarity_scores.sum()
        )

    # Get TF-IDF vectors of feedback documents
    feedback_matrix = tfidf_matrix[top_indices]

    # Aggregate weighted TF-IDF scores
    term_scores = np.asarray(
        feedback_weights @ feedback_matrix
    ).ravel()

    # Get vocabulary
    feature_names = tfidf_vectorizer.get_feature_names_out()

    # Exclude original query terms
    query_terms = set(query_tokens)

    candidates = []

    for idx in np.argsort(term_scores)[::-1]:
        term = feature_names[idx]
        score = term_scores[idx]

        if score <= 0 or term in query_terms:
            continue

        candidates.append({
            "term": term,
            "score": score
        })

        if len(candidates) >= top_n:
            break

    return pd.DataFrame(candidates)

In [33]:

def search_articles(
    query,
    top_k=10,
    use_query_expansion=False,
    expansion_top_n=5
):
    # Preprocess original query
    query_data = preprocess_query(query)
    processed_query = query_data["processed_query"]
    query_tokens = query_data["filtered_tokens"]

    if not processed_query:
        print("Query menjadi kosong setelah preprocessing.")
        return

    # Initial retrieval using original query
    query_vector = vectorizer.transform([processed_query])

    if query_vector.nnz == 0:
        print("Tidak ada term query yang ditemukan dalam vocabulary TF-IDF.")
        return

    initial_scores = cosine_similarity(
        query_vector, matrix
    ).ravel()

    initial_ranked_indices = np.argsort(initial_scores)[::-1]
    initial_ranked_indices = [
        idx for idx in initial_ranked_indices
        if initial_scores[idx] > 0
    ][:top_k]

    if not initial_ranked_indices:
        print("Tidak ditemukan dokumen yang relevan.")
        return

    # Initialize query expansion variables
    expansion_terms = []
    expanded_query = processed_query

    # Apply Pseudo-Relevance Feedback
    if use_query_expansion:
        expansion_candidates = extract_expansion_terms(
            tfidf_matrix=matrix,
            tfidf_vectorizer=vectorizer,
            top_indices=initial_ranked_indices,
            similarity_scores=initial_scores[initial_ranked_indices],
            query_tokens=query_tokens,
            top_n=expansion_top_n
        )

        expansion_terms = expansion_candidates["term"].tolist()

        # Construct expanded query
        expanded_query_tokens = list(
            dict.fromkeys(query_tokens + expansion_terms)
        )

        expanded_query = " ".join(expanded_query_tokens)

        # Transform expanded query
        query_vector = vectorizer.transform([expanded_query])

    # Final retrieval
    scores = cosine_similarity(query_vector, matrix).ravel()

    ranked_indices = np.argsort(scores)[::-1]
    ranked_indices = [
        idx for idx in ranked_indices
        if scores[idx] > 0
    ][:top_k]

    if not ranked_indices:
        print("Tidak ditemukan dokumen dengan skor cosine similarity di atas nol.")
        return

    # Build results
    results = df.iloc[ranked_indices].copy()

    results.insert(0, "rank", np.arange(1, len(results) + 1))
    results.insert(1, "doc_index", ranked_indices)
    
    results["similarity_score"] = scores[ranked_indices]

    results = (
        results
        .sort_values("similarity_score", ascending=False)
        .reset_index(drop=True)
    )

    # Display retrieval information
    print(f"Query            : {query}")
    print(f"Processed Query  : {processed_query}")
    print(f"Query Expansion  : {'Enabled' if use_query_expansion else 'Disabled'}")

    if use_query_expansion:
        print(f"Expansion Terms  : {', '.join(expansion_terms)}")
        print(f"Expanded Query   : {expanded_query}")

    return results


In [34]:
query = "dampak mbg terhadap pelajar"
results = search_articles(
  query,
  top_k = 10,
  use_query_expansion = True,
  expansion_top_n = 5
)

# Show all results without truncation
# pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_colwidth", 500)
display(results)

Query            : dampak mbg terhadap pelajar
Processed Query  : dampak mbg pelajar
Query Expansion  : Enabled
Expansion Terms  : program, bgn, kediri, keracunan, siswa
Expanded Query   : dampak mbg pelajar program bgn kediri keracunan siswa


,rank,doc_index,link,date,title,content,keywords,category,normalized_title,normalized_content,normalized_keywords,normalized,tokens,filtered_tokens,similarity_score
0,1,926,https://surabaya.kompas.com/read/2026/07/28/141500378/kasus-keracunan-mbg-di-kediri--211-siswa-terdampak-2-dirawat-dpr-minta,2026-07-28 14:15:00,"Kasus Keracunan MBG di Kediri: 211 Siswa Terdampak, 2 Dirawat, DPR Minta Katering Dihentikan Sementara","KEDIRI, KOMPAS.com– Kasus dugaan keracunan makanan yang menimpa ratusan pelajar di Kabupaten Kediri, Jawa Timur, usai menyantap menu program Makan Bergizi Gratis (MBG) mendapat sorotan tajam dari Dewan Perwakilan Rakyat (DPR) RI. Dinas Kesehatan (Dinkes) Kabupaten Kediri mencatat, sebanyak 211 pelajar dari SMP Negeri 1 Kras dan SD Negeri 1 Purwodadi melaporkan mengalami gangguan kesehatan usai mengonsumsi makanan yang disajikan oleh Satuan Pelayanan Pemenuhan Gizi (SPPG) Purwodadi pada Kamis...","madiun, keracunan mbg, korban keracunan mbg, sppg purwodadi, keracunan makan bergizi gratis kediri, keracunan mbg kediri, smpn 1 kras kediri",Regional,kasus keracunan mbg di kediri siswa terdampak dirawat dpr minta katering dihentikan sementara,kediri kasus dugaan keracunan makanan yang menimpa ratusan pelajar di kabupaten kediri jawa timur usai menyantap menu program makan bergizi gratis mbg mendapat sorotan tajam dari dewan perwakilan rakyat dpr ri dinas kesehatan dinkes kabupaten kediri mencatat sebanyak pelajar dari smp negeri kras dan sd negeri purwodadi melaporkan mengalami gangguan kesehatan usai mengonsumsi makanan yang disajikan oleh satuan pelayanan pemenuhan gizi sppg purwodadi pada kamis gejala keracunan makanan seperti...,madiun keracunan mbg korban keracunan mbg sppg purwodadi keracunan makan bergizi gratis kediri keracunan mbg kediri smpn kras kediri,kasus keracunan mbg di kediri siswa terdampak dirawat dpr minta katering dihentikan sementara kediri kasus dugaan keracunan makanan yang menimpa ratusan pelajar di kabupaten kediri jawa timur usai menyantap menu program makan bergizi gratis mbg mendapat sorotan tajam dari dewan perwakilan rakyat dpr ri dinas kesehatan dinkes kabupaten kediri mencatat sebanyak pelajar dari smp negeri kras dan sd negeri purwodadi melaporkan mengalami gangguan kesehatan usai mengonsumsi makanan yang disajikan o...,"['kasus', 'keracunan', 'mbg', 'di', 'kediri', 'siswa', 'terdampak', 'dirawat', 'dpr', 'minta', 'katering', 'dihentikan', 'sementara', 'kediri', 'kasus', 'dugaan', 'keracunan', 'makanan', 'yang', 'menimpa', 'ratusan', 'pelajar', 'di', 'kabupaten', 'kediri', 'jawa', 'timur', 'usai', 'menyantap', 'menu', 'program', 'makan', 'bergizi', 'gratis', 'mbg', 'mendapat', 'sorotan', 'tajam', 'dari', 'dewan', 'perwakilan', 'rakyat', 'dpr', 'ri', 'dinas', 'kesehatan', 'dinkes', 'kabupaten', 'kediri', 'men...","['keracunan', 'mbg', 'kediri', 'siswa', 'terdampak', 'dirawat', 'dpr', 'katering', 'dihentikan', 'kediri', 'dugaan', 'keracunan', 'makanan', 'menimpa', 'ratusan', 'pelajar', 'kabupaten', 'kediri', 'jawa', 'timur', 'menyantap', 'menu', 'program', 'makan', 'bergizi', 'gratis', 'mbg', 'sorotan', 'tajam', 'dewan', 'perwakilan', 'rakyat', 'dpr', 'ri', 'dinas', 'kesehatan', 'dinkes', 'kabupaten', 'kediri', 'mencatat', 'pelajar', 'smp', 'negeri', 'kras', 'sd', 'negeri', 'purwodadi', 'melaporkan', '...",0.578690
1,2,745,https://surabaya.kompas.com/read/2026/07/29/095925578/dugaan-keracunan-mbg-di-kediri-dinkes-bicara-kondisi-siswa-hingga-sampel,2026-07-29 09:59:25,"Dugaan Keracunan MBG di Kediri, Dinkes Bicara Kondisi Siswa hingga Sampel Makanan","KEDIRI, KOMPAS.com- Kepala Dinas Kesehatan Kabupaten Kediri, Jawa Timur, Ahmad Khotib, memastikan seluruh siswa yang sempat mengalami gejala keracunan diduga usai menyantap menu Makan Bergizi Gratis (MBG) di Kecamatan Kras, kondisinya kini sudah mulai membaik. Meski demikian, langkah-langkah penelusuran penyebab keracunannya tetap dilakukan. Termasuk memeriksakan sampel makanan ke laboratorium di tingkat provinsi untuk mengetahui kandungan